# Epistemic Chain Complex: Information Exchange & Belief Updates

This notebook implements a discrete model of an **Epistemic Chain Complex** to simulate how information exchange and context-dependent attention influence the belief updates of multiple agents.

## 1. Theoretical Framework

We model a system of $N=3$ agents (Alice, Bob, and Charlie) evaluating their self-image under the proposition:
$$p = \text{"I am handsome"}$$
across $M=3$ distinct contexts:
1.  **Context $c_1$:** "In comparison with Abraham Lincoln"
2.  **Context $c_2$:** "In front of a mirror"
3.  **Context $c_3$:** "In my best selfie"

### The Belief Matrix
The context-specific beliefs of the agents are represented by an $N \times M$ matrix $A$, where $A_{a, c} \in [0, 1]$ represents agent $a$'s belief score in context $c$. 

The **generalized belief** $B_a(t)$ of agent $a$ is the attention-weighted average of their context-specific beliefs:
$$B_a(t) = \sum_{c=1}^M A_{a, c}(t) w_c(t)$$
where $w(t) = [w_1(t), \dots, w_M(t)]^T$ is the attention weight vector for the contexts, which evolves dynamically based on the level of disagreement (controversy) in each context.

### The Simplicial Complex
The social network of the agents is represented as a simplicial complex:
*   **0-Simplices (Vertices $D_0$):** Spanned by the agents $\{v_1, v_2, v_3\}$. Represents individual belief states.
*   **1-Simplices (Oriented Edges $D_1$):** Spanned by communication channels $\{e_{12}, e_{23}, e_{31}\}$. Represents information flow/disagreement.
*   **2-Simplex (Face $D_2$):** Spanned by the group context $\{f_{123}\}$. Represents group consensus.

The chain complex is:
$$D_2 \xrightarrow{\partial_2} D_1 \xrightarrow{\partial_1} D_0$$

The boundary matrices are defined as:
$$\partial_1 = \begin{pmatrix} -1 & 0 & 1 \\ 1 & -1 & 0 \\ 0 & 1 & -1 \end{pmatrix}, \quad \partial_2 = \begin{pmatrix} 1 \\ 1 \\ 1 \end{pmatrix}$$
satisfying the nilpotency condition:
$$\partial_1 \circ \partial_2 = 0$$
which mathematically defines how circular arguments or **echo chambers** produce zero net belief updates.

## 2. Epistemic Curvature Formulation

To measure the stability and uniformity of beliefs, we define the **Epistemic Curvature** using the Laplacians of the social graph and the context space.

### A. Social Epistemic Curvature ($K_{\text{social}}$)
For each context $c$, the social curvature measures the degree to which agent $a$'s belief deviates from the local consensus of their peers. It is calculated using the social graph Laplacian $\mathbf{L}_{\text{social}} = \partial_1 \partial_1^T$:
$$\mathbf{K}_{\text{social}}(t) = \mathbf{L}_{\text{social}} A_{\bullet, c}(t)$$
If an agent agrees with their neighbors, the social curvature at that vertex is $0$.

### B. Cognitive Epistemic Curvature ($K_{\text{cognitive}}$)
For each agent $a$, the cognitive curvature measures the variation or "non-uniformity" of their beliefs across different contexts (a measure of cognitive dissonance or context-sensitivity). It is calculated using the context graph Laplacian $\mathbf{L}_{\text{context}}$:
$$\mathbf{K}_{\text{cognitive}}(t) = A_{a, \bullet}(t) \mathbf{L}_{\text{context}}$$
If an agent holds the exact same belief regardless of context (flat belief space), their cognitive curvature is $0$.

### C. Total Epistemic Curvature ($K_{\text{total}}$)
The total epistemic curvature of the system is the combined divergence across both social and contextual dimensions (modeled as the Kronecker sum Laplacian on the product space $\text{Graph} \times \text{Contexts}$):
$$\mathbf{K}_{\text{total}}(t) = \mathbf{L}_{\text{social}} A(t) + A(t) \mathbf{L}_{\text{context}}$$
This joint curvature measures the total "torsion" or failure of global flattening in the agent-context belief sheaf. As the system converges toward consensus and context-equivalence, $\mathbf{K}_{\text{total}} \to 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class EpistemicComplex:
    def __init__(self, A_init, w_init, kappa=0.4, gamma=0.5, epsilon=0.05):
        """
        Initialize the Epistemic Complex.
        
        Parameters:
        A_init  : 2D array (N x M), initial context-specific beliefs
        w_init  : 1D array (M), initial attention weights
        kappa   : float, belief diffusion learning rate
        gamma   : float, attention adaptation rate
        epsilon : float, baseline attention salience
        """
        self.A = np.array(A_init, dtype=float)
        self.w = np.array(w_init, dtype=float)
        self.w /= np.sum(self.w) # Ensure normalization
        
        self.kappa = kappa
        self.gamma = gamma
        self.epsilon = epsilon
        
        # 3-agent simplicial complex boundary operators
        # d1 maps 1-chains (edges) to 0-chains (vertices)
        self.d1 = np.array([
            [-1,  0,  1],
            [ 1, -1,  0],
            [ 0,  1, -1]
        ], dtype=float)
        
        # d2 maps 2-chains (faces) to 1-chains (edges)
        self.d2 = np.array([[1], [1], [1]], dtype=float)
        
        # Social graph Laplacian L_social = d1 * d1.T
        self.L_social = np.dot(self.d1, self.d1.T)
        
        # Context space Laplacian L_context (fully connected context network)
        self.L_context = np.array([
            [ 2, -1, -1],
            [-1,  2, -1],
            [-1, -1,  2]
        ], dtype=float)
        
    def get_generalized_beliefs(self):
        """Calculate B = A * w"""
        return np.dot(self.A, self.w)
        
    def calculate_social_curvature(self):
        """K_social = L_social * A (for each context column)"""
        return np.dot(self.L_social, self.A)
        
    def calculate_cognitive_curvature(self):
        """K_cognitive = A * L_context (for each agent row)"""
        return np.dot(self.A, self.L_context)
        
    def calculate_total_curvature(self):
        """K_total = L_social * A + A * L_context"""
        return self.calculate_social_curvature() + self.calculate_cognitive_curvature()
        
    def update(self, dt=0.01):
        """Update belief matrix A and attention weights w by one time step"""
        # 1. Update beliefs in each context via graph diffusion
        # dA/dt = -kappa * L_social * A
        dA_dt = -self.kappa * np.dot(self.L_social, self.A)
        self.A += dA_dt * dt
        
        # 2. Update attention weights based on context salience
        salience = np.zeros(3)
        for c in range(3):
            diff_01 = np.abs(self.A[1, c] - self.A[0, c])
            diff_12 = np.abs(self.A[2, c] - self.A[1, c])
            diff_20 = np.abs(self.A[0, c] - self.A[2, c])
            salience[c] = diff_01 + diff_12 + diff_20 + self.epsilon
            
        normalized_salience = salience / np.sum(salience)
        dw_dt = self.gamma * (normalized_salience - self.w)
        self.w += dw_dt * dt
        
        # Ensure weights are normalized
        self.w = np.clip(self.w, 0, 1)
        self.w /= np.sum(self.w)

In [ ]:
# Initialize agents and contexts
agents = ["Alice", "Bob", "Charlie"]
contexts = ["Lincoln Comparison", "Mirror View", "Best Selfie"]

# Initial belief matrix A (rows=agents, cols=contexts)
A_init = np.array([
    [0.9, 0.5, 0.8],  # Alice
    [0.8, 0.3, 0.9],  # Bob
    [0.7, 0.6, 0.7]   # Charlie
])

# Initial context weights w (equally weighted)
w_init = np.array([1/3, 1/3, 1/3])

# Create the complex
model = EpistemicComplex(A_init, w_init, kappa=0.4, gamma=0.5, epsilon=0.05)

# Simulation time parameters
T = 15.0
dt = 0.01
steps = int(T / dt)

# History trackers
time_history = np.linspace(0, T, steps)
A_history = np.zeros((steps, 3, 3))
w_history = np.zeros((steps, 3))
B_history = np.zeros((steps, 3))
K_social_history = np.zeros((steps, 3, 3))
K_cognitive_history = np.zeros((steps, 3, 3))
K_total_history = np.zeros((steps, 3, 3))

# Run the simulation loop
for i in range(steps):
    A_history[i] = model.A.copy()
    w_history[i] = model.w.copy()
    B_history[i] = model.get_generalized_beliefs()
    K_social_history[i] = model.calculate_social_curvature()
    K_cognitive_history[i] = model.calculate_cognitive_curvature()
    K_total_history[i] = model.calculate_total_curvature()
    
    model.update(dt)

print(f"Simulation completed successfully over {T} seconds ({steps} steps).")

In [ ]:
# Define visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig = plt.figure(figsize=(15, 11), dpi=150)
grid = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.25)

# Color palettes
colors_agents = ['#E91E63', '#00BCD4', '#4CAF50'] # Pink, cyan, green
colors_contexts = ['#9C27B0', '#FF9800', '#FFEB3B'] # Purple, orange, yellow
colors_curvatures = ['#FF5722', '#3F51B5', '#9E9E9E'] # Orange-red, indigo, gray

# Panel 1: Context-Specific Beliefs A_ac
ax1 = fig.add_subplot(grid[0, 0])
ax1.set_title("Evolution of Context-Specific Beliefs $A_{a,c}$", fontsize=12, pad=12, fontweight='bold')
linestyle = ['-', '--', ':']
for a in range(3):
    for c in range(3):
        ax1.plot(time_history, A_history[:, a, c], 
                 color=colors_agents[a], linestyle=linestyle[c], alpha=0.8, linewidth=1.5)
ax1.set_xlabel("Time", fontsize=10)
ax1.set_ylabel("Belief Value (0 to 1)", fontsize=10)

# Custom legend
from matplotlib.lines import Line2D
legend_elements_1 = [
    Line2D([0], [0], color=colors_agents[0], lw=2, label='Alice'),
    Line2D([0], [0], color=colors_agents[1], lw=2, label='Bob'),
    Line2D([0], [0], color=colors_agents[2], lw=2, label='Charlie'),
    Line2D([0], [0], color='#555555', lw=1.5, ls='-', label='Lincoln'),
    Line2D([0], [0], color='#555555', lw=1.5, ls='--', label='Mirror'),
    Line2D([0], [0], color='#555555', lw=1.5, ls=':', label='Selfie'),
]
ax1.legend(handles=legend_elements_1, loc='lower right', frameon=True, fontsize=8)

# Panel 2: Context Attention Weights w_c
ax2 = fig.add_subplot(grid[0, 1])
ax2.set_title("Evolution of Context Attention Weights $w_c$", fontsize=12, pad=12, fontweight='bold')
for c in range(3):
    ax2.plot(time_history, w_history[:, c], label=contexts[c], color=colors_contexts[c], linewidth=2.5)
ax2.set_xlabel("Time", fontsize=10)
ax2.set_ylabel("Attention Weight", fontsize=10)
ax2.legend(loc='upper right', frameon=True, fontsize=9)

# Panel 3: Generalized Beliefs B_a
ax3 = fig.add_subplot(grid[1, 0])
ax3.set_title(r"Evolution of Generalized Beliefs $B_a = \sum_c A_{a,c} w_c$", fontsize=12, pad=12, fontweight='bold')
for a in range(3):
    ax3.plot(time_history, B_history[:, a], label=agents[a], color=colors_agents[a], linewidth=2.5)
ax3.set_xlabel("Time", fontsize=10)
ax3.set_ylabel("Generalized Belief", fontsize=10)
ax3.legend(loc='lower right', frameon=True, fontsize=9)

# Panel 4: Epistemic Curvatures (Frobenius Norms)
ax4 = fig.add_subplot(grid[1, 1])
ax4.set_title("Decay of Epistemic Curvatures (Frobenius Norms)", fontsize=12, pad=12, fontweight='bold')

norm_social = np.linalg.norm(K_social_history, axis=(1, 2))
norm_cognitive = np.linalg.norm(K_cognitive_history, axis=(1, 2))
norm_total = np.linalg.norm(K_total_history, axis=(1, 2))

ax4.plot(time_history, norm_social, label="Social Curvature ||$K_{social}$||", color=colors_curvatures[0], linewidth=2)
ax4.plot(time_history, norm_cognitive, label="Cognitive Curvature ||$K_{cognitive}$||", color=colors_curvatures[1], linewidth=2)
ax4.plot(time_history, norm_total, label="Total Curvature ||$K_{total}$||", color=colors_curvatures[2], linewidth=2.5, linestyle='-.')

ax4.set_xlabel("Time", fontsize=10)
ax4.set_ylabel("Curvature Magnitude (Frobenius Norm)", fontsize=10)
ax4.legend(loc='upper right', frameon=True, fontsize=9)

plt.tight_layout()
plt.show()

## 3. Analysis & Homological Interpretation

### A. Interpretation of the Curvature Decay
As observed in the bottom-right plot:
1.  **Social Curvature Decay:** The social curvature $K_{\text{social}}$ decays toward $0$ because of the graph diffusion process. The agents influence each other, leading to a consensus in each of the three contexts. In each context $c$, the belief vector $A_{\bullet, c}$ becomes a constant vector (a $0$-cycle in the homology), making the Laplacian product $\mathbf{L}_{\text{social}} A = 0$.
2.  **Cognitive Epistemic Curvature:** The cognitive curvature $K_{\text{cognitive}}$ measures the differences in beliefs across contexts. In our simulation, the beliefs converge to a consensus, but the consensus values are *different* for each context (e.g. $0.8$ for selfie, $0.47$ for mirror). Because of this, cognitive curvature remains non-zero unless the contexts themselves are structurally unified (or the agent down-weights the context differences). This illustrates that even when social agreement is reached, an agent can still experience internal cognitive complexity or context-sensitivity.
3.  **Total Curvature:** The total curvature $K_{\text{total}}$ stabilizes at a non-zero value, representing the persistent cognitive structural difference between the contexts.

### B. The Homological "Echo Chamber"
In our chain complex $D_2 \xrightarrow{\partial_2} D_1 \xrightarrow{\partial_1} D_0$, the fact that $\partial_1 \circ \partial_2 = 0$ carries a physical/social interpretation:
*   Any flow $J \in D_1$ that is in the image of $\partial_2$ represents a **circular argument** or an **echo chamber** ($J_{\text{circ}} = \alpha [1, 1, 1]^T$).
*   The update equation is:
    $$\frac{dB}{dt} = - \kappa \cdot \partial_1(J)$$
*   For an echo chamber, the update rate is:
    $$\frac{dB}{dt} = - \kappa \cdot \partial_1(\partial_2(\alpha)) = 0$$
*   **Result:** A closed cycle of information flow does not change the net belief state of the group. Only flows that do not form closed loops ($J \notin \text{im } \partial_2$) can drive updates and shift the generalized beliefs.

### C. Change of Basis vs. Consensus
When an agent experiences deep cognitive dissonance (large internal cognitive curvature), they have two options:
1.  **Consensus Alignment (Option A):** Conform to the group's transition maps, reducing the Čech obstruction to zero.
2.  **Change of Basis (Option B):** Apply an isomorphism to their belief representation module $D_n' = \psi D_n \psi^{-1}$ to construct a new internal model. This allows them to maintain their private beliefs while maintaining local compatibility with observations, effectively moving the curvature out of their active workspace.